In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().parent
ENV_LOADED = load_dotenv(ROOT / ".env")

REQUIRED_KEYS = (
    "OPENAI_API_KEY",
    "ANTHROPIC_API_KEY",
    "TYPESAFE_API_KEY",
    "LANGFUSE_PUBLIC_KEY",
    "LANGFUSE_SECRET_KEY",
    "LANGFUSE_BASE_URL",
    "OPENAI_MODEL",
    "ANTHROPIC_MODEL",
)
missing = [key for key in REQUIRED_KEYS if not os.getenv(key)]

print(f"Environment loaded: {ENV_LOADED}")
print(f"Root directory: {ROOT}")
print(f"Missing keys: {missing}")

assert ROOT.name == "Techstack.io", f"ROOT is wrong: {ROOT}"
assert not missing, f"fill these in .env {missing}"

Environment loaded: True
Root directory: /Users/michaelperrine/Techstack.io
Missing keys: []


In [ ]:
from langfuse import get_client

langfuse = get_client()
langfuse_ok = langfuse.auth_check()
assert langfuse_ok, (
    "Langfuse auth failed: check LANGFUSE_PUBLIC_KEY, "
    "LANGFUSE_SECRET_KEY and LANGFUSE_BASE_URL (uS vs EU region)"
)
print(f"Langfuse client initialized: {langfuse_ok}")

Langfuse client initialized: True


In [15]:
from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI
from langfuse.langchain import CallbackHandler

PROMPT = "Reply with exactly: OK"

handler = CallbackHandler()
models = {
    "primary": ChatOpenAI(
        model=os.getenv("OPENAI_MODEL"), temperature=1
    ),  # Primary model only accepts temperature=1
    "checker": ChatAnthropic(model=os.getenv("ANTHROPIC_MODEL")),
}

replies = {}
for name, llm in models.items():
    reply = llm.invoke(PROMPT, config={"callbacks": [handler]})
    replies[name] = reply.text.strip()
    print(f"{name} -> {replies[name]}")

langfuse.flush()

assert all(r == "OK" for r in replies.values()), f"Unexpected: {replies}"

primary -> OK
checker -> OK
